# L4: Why Context Matters

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

![Why Context Matters](L4.png)

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
pio.renderers.default = 'png'

load_dotenv()

if not os.getenv('OPENAI_API_KEY'):
    print('WARNING: OPENAI_API_KEY not set.')
else:
    print('✓ OpenAI API key loaded successfully')

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>
</div>

## Section 1: The Problem

In [ ]:
from src.data.sample_prs import load_pr

pr = load_pr('pr_001_auth_bypass')

expected_issues_text = "\n".join(
    f"  • {issue}" for issue in pr['expected_issues'])
print(f"PR Title: {pr['title']}\n\nDiff:\n{pr['diff']}"
    f"\n\nExpected Issues:\n{expected_issues_text}")

## Section 2: The Review System

In [ ]:
from src.review_agents.reviewer import review, DEFAULT_REVIEW_PROMPT

separator = "=" * 80
print(f"{separator}\nREVIEW PROMPT (for BOTH diff-only & context-aware)\n"
      f"{separator}\n{DEFAULT_REVIEW_PROMPT}\n{separator}")

### Single Example: Diff-Only Review

In [ ]:
diff_only_review = review(pr, context=None, task_context=None)

findings_text = "\n\n".join(
    f"{i}. {finding['issue']}\n   Severity: {finding['severity']}"
    for i, finding in enumerate(diff_only_review['findings'], 1)
)
print(f"Diff-Only Review Findings:\n\n{findings_text}\n\n"
      f"Total: {len(diff_only_review['findings'])} issue(s) found")

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

In [ ]:
from src.utils.display import create_detailed_matching_table

diff_only_matching = create_detailed_matching_table(diff_only_review, 
                                                    pr['expected_issues'])
display(diff_only_matching)

### Single Example: Context-Aware Review

In [ ]:
context_aware_review = review(pr, context=pr.get('context', ''), 
                              task_context=pr.get('task_context', ''))

findings_text = "\n\n".join(
    f"{i}. {finding['issue']}\n   Severity: {finding['severity']}"
    for i, finding in enumerate(context_aware_review['findings'], 1))
print(
    f"Context-Aware Review Findings:\n\n{findings_text}\n\n"
    f"Total: {len(context_aware_review['findings'])} issue(s) found\n\n"
    "💡 The ONLY difference: we passed context + task_context")

In [ ]:
context_aware_matching = create_detailed_matching_table(
    context_aware_review, pr['expected_issues'])
display(context_aware_matching)

In [ ]:
from src.utils.display import create_comparison_table

comparison = create_comparison_table(diff_only_review, context_aware_review, 
    pr['expected_issues'])

display(comparison)

## Section 3: Full Evaluation

In [ ]:
from src.data.sample_prs import load_all_prs

all_prs = load_all_prs()
print(f'Loaded {len(all_prs)} PRs for evaluation')

In [ ]:
from src.utils.evaluation import evaluate_reviewers

print("Evaluating both approaches across all PRs...\n"
      "This may take a few minutes...\n")

results = evaluate_reviewers(all_prs)

print(f"\n{'=' * 80}\nEVALUATION COMPLETE\n{'=' * 80}")

### Benchmark Metrics Comparison

In [ ]:
from src.utils.display import build_metrics_dataframe, print_metrics_summary

metrics_df = build_metrics_dataframe(results)
display(metrics_df)

print_metrics_summary(results)

In [ ]:
from src.utils.plotting import create_metrics_chart

fig = create_metrics_chart(results)
fig.show()

## Section 4: Detailed Analysis

In [ ]:
from src.utils.analysis import analyze_findings, display_findings_summary

findings_summary = analyze_findings(results)
display_findings_summary(findings_summary)